In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/steam-dataset-2025-multi-modal-gaming-analytics/steam-dataset-2025-v1/DATASET_CARD.md
/kaggle/input/steam-dataset-2025-multi-modal-gaming-analytics/steam-dataset-2025-v1/steam-dataset-2025-full-schema.sql
/kaggle/input/steam-dataset-2025-multi-modal-gaming-analytics/steam-dataset-2025-v1/README.md
/kaggle/input/steam-dataset-2025-multi-modal-gaming-analytics/steam-dataset-2025-v1/DATA_DICTIONARY.md
/kaggle/input/steam-dataset-2025-multi-modal-gaming-analytics/steam-dataset-2025-v1/notebook-data/README.md
/kaggle/input/steam-dataset-2025-multi-modal-gaming-analytics/steam-dataset-2025-v1/notebook-data/03-the-semantic-fingerprint/03-the-semantic-fingerprint-preview.csv
/kaggle/input/steam-dataset-2025-multi-modal-gaming-analytics/steam-dataset-2025-v1/notebook-data/03-the-semantic-fingerprint/03-the-semantic-fingerprint.parquet
/kaggle/input/steam-dataset-2025-multi-modal-gaming-analytics/steam-dataset-2025-v1/notebook-data/03-the-semantic-fingerprint/.gitattributes
/kaggle

Load Core Tables

In [14]:
base_path = "/kaggle/input/steam-dataset-2025-multi-modal-gaming-analytics/steam_dataset_2025_csv_package_v1/steam_dataset_2025_csv/"

applications = pd.read_csv(base_path + "applications.csv")
reviews = pd.read_csv(base_path + "reviews.csv")
genres = pd.read_csv(base_path + "genres.csv")
application_genres = pd.read_csv(base_path + "application_genres.csv")
platforms = pd.read_csv(base_path + "platforms.csv")
publishers = pd.read_csv(base_path + "publishers.csv")
application_publishers = pd.read_csv(base_path + "application_publishers.csv")


/tmp/ipykernel_55/3439876041.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  applications = pd.read_csv(base_path + "applications.csv")


Validate Row Counts & Schema

In [15]:
tables = {
    "applications": applications,
    "reviews": reviews,
    "genres": genres,
    "application_genres": application_genres,
    "platforms": platforms,
    "publishers": publishers,
    "application_publishers": application_publishers
}

summary = []

for name, df in tables.items():
    summary.append({
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

pd.DataFrame(summary)


,table,rows,columns
0,applications,239664,30
1,reviews,1048148,23
2,genres,154,2
3,application_genres,587515,2
4,platforms,3,2
5,publishers,85699,2
6,application_publishers,223048,2


Missing Value Percentage per Table

In [16]:
missing_summary = []

for name, df in tables.items():
    missing_pct = (df.isnull().sum() / len(df)) * 100
    missing_summary.append(
        missing_pct.round(2).to_frame(name="missing_%").assign(table=name)
    )

missing_df = pd.concat(missing_summary)
missing_df.reset_index().rename(columns={"index": "column"})


,column,missing_%,table
0,appid,0.00,applications
1,name,0.00,applications
2,type,1.11,applications
3,is_free,0.00,applications
4,release_date,15.38,applications
...,...,...,...
58,name,0.00,platforms
59,id,0.00,publishers
60,name,0.01,publishers
61,appid,0.00,application_publishers


Primary–Foreign Key Validation

In [17]:
invalid_review_appids = reviews.loc[
    ~reviews['appid'].isin(applications['appid']),
    'appid'
].nunique()

invalid_review_appids


0

All reviews correctly reference valid application IDs.
No orphan reviews were detected.

AppID ↔ Genres Relationship

In [18]:
invalid_genre_appids = application_genres.loc[
    ~application_genres['appid'].isin(applications['appid']),
    'appid'
].nunique()

invalid_genre_appids



0

Application–genre mappings are consistent, with no invalid application references.

Genre ID Validation

In [21]:
application_genres.columns


Index(['appid', 'genre_id'], dtype='object')

In [22]:
genres.columns


Index(['id', 'name'], dtype='object')

In [24]:
invalid_genre_ids = application_genres.loc[
    ~application_genres['genre_id'].isin(genres['id']),
    'genre_id'
].nunique()

invalid_genre_ids


0

Publisher ID Validation

In [30]:
invalid_publisher_ids = application_publishers.loc[
    ~application_publishers['publisher_id'].isin(publishers['id']),
    'publisher_id'
].nunique()

invalid_publisher_ids


0

In [28]:
application_publishers.columns


Index(['appid', 'publisher_id'], dtype='object')

In [29]:
publishers.columns


Index(['id', 'name'], dtype='object')

In [31]:
invalid_appids_publishers = application_publishers.loc[
    ~application_publishers['appid'].isin(applications['appid']),
    'appid'
].nunique()

invalid_appids_publishers


0

In [2]:
reviews_path = "/kaggle/input/steam-dataset-2025-multi-modal-gaming-analytics/steam_dataset_2025_csv_package_v1/steam_dataset_2025_csv/reviews.csv"
df = pd.read_csv(reviews_path)

df.head()


,recommendationid,appid,author_steamid,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played,language,...,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,created_at,updated_at
0,10000000,264220,76561198085405844,760,74,12.0,0.0,12.0,1.399060e+09,polish,...,True,0,1,0.459906,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00
1,100001066,1006440,76561198014439859,485,234,424.0,0.0,424.0,1.632666e+09,russian,...,True,8,0,0.630770,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00
2,100002344,320721,76561198048038590,0,385,0.0,0.0,NaN,0.000000e+00,german,...,False,1,0,0.523810,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00
3,100002361,1604700,76561197994386273,0,3,86.0,0.0,86.0,1.632674e+09,english,...,True,3,0,0.541985,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00
4,100002504,1338560,76561198138996331,816,26,25.0,0.0,25.0,1.632639e+09,english,...,True,1,0,0.523810,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00


In [3]:
df.shape


(1048148, 23)

In [4]:
df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048148 entries, 0 to 1048147
Data columns (total 23 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   recommendationid                1048148 non-null  int64  
 1   appid                           1048148 non-null  int64  
 2   author_steamid                  1048148 non-null  int64  
 3   author_num_games_owned          1048148 non-null  int64  
 4   author_num_reviews              1048148 non-null  int64  
 5   author_playtime_forever         1048145 non-null  float64
 6   author_playtime_last_two_weeks  1048145 non-null  float64
 7   author_playtime_at_review       869291 non-null   float64
 8   author_last_played              1048145 non-null  float64
 9   language                        1048148 non-null  object 
 10  review_text                     1047369 non-null  object 
 11  timestamp_created               1048148 non-null  int64  
 12  

In [5]:
df.columns


Index(['recommendationid', 'appid', 'author_steamid', 'author_num_games_owned',
       'author_num_reviews', 'author_playtime_forever',
       'author_playtime_last_two_weeks', 'author_playtime_at_review',
       'author_last_played', 'language', 'review_text', 'timestamp_created',
       'timestamp_updated', 'voted_up', 'votes_up', 'votes_funny',
       'weighted_vote_score', 'comment_count', 'steam_purchase',
       'received_for_free', 'written_during_early_access', 'created_at',
       'updated_at'],
      dtype='object')

In [6]:
df.isnull().sum().sort_values(ascending=False)


author_playtime_at_review         178857
review_text                          779
author_playtime_last_two_weeks         3
author_playtime_forever                3
author_last_played                     3
author_steamid                         0
recommendationid                       0
author_num_games_owned                 0
author_num_reviews                     0
appid                                  0
language                               0
timestamp_created                      0
timestamp_updated                      0
voted_up                               0
votes_up                               0
votes_funny                            0
weighted_vote_score                    0
comment_count                          0
steam_purchase                         0
received_for_free                      0
written_during_early_access            0
created_at                             0
updated_at                             0
dtype: int64

In [7]:
df.duplicated().sum()


np.int64(0)

Univariate Analysis

In [9]:
df['author_playtime_at_review'].describe()


count    8.692910e+05
mean     1.590649e+03
std      1.325358e+04
min      1.000000e+00
25%      5.100000e+01
50%      1.940000e+02
75%      6.610000e+02
max      2.030768e+06
Name: author_playtime_at_review, dtype: float64

In [10]:
df['review_length'] = df['review_text'].str.len()
df['review_length'].describe()


count    1.047369e+06
mean     3.591773e+02
std      6.752051e+02
min      1.000000e+00
25%      4.500000e+01
50%      1.370000e+02
75%      3.720000e+02
max      8.000000e+03
Name: review_length, dtype: float64

In [11]:
df['language'].value_counts().head(10)


language
english      556231
schinese     136210
russian      101642
german        36859
brazilian     32318
japanese      27894
spanish       27285
french        25090
koreana       24859
turkish       17949
Name: count, dtype: int64

In [13]:
df['author_playtime_at_review'].quantile([0.95, 0.99])


0.95     4536.5
0.99    22801.2
Name: author_playtime_at_review, dtype: float64